In [ ]:
import pandas as pd, numpy as np
from lsff_utils import data_processing

In [ ]:
location = "nigeria"
directory = "/snfs1/DATA/DHS_PROG_DHS/NGA/2018/"
results_dir = "../results"

### WRA

In [ ]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = pd.read_stata(
    directory
    + (
        "IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA"
        if location == "india"
        else "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA"
    ),
    columns=wra_columns.keys(),
)

In [ ]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [ ]:
wra_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    wra_data.wealth_quintile
)

In [ ]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Siblings

Sibling survival data appears to only be available as a kind of side table-within-a-table on WRA.

It is labeled "MM" because it is used to calculate maternal mortality (among other things).

In [ ]:
respondent_column_names = {
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "v005": "weight",
}

# These are suffixed with an underscore and an integer, e.g. mm1_01
sibling_column_names = {
    # MM1                    Sex of sibling                                  7156    1    N    I   20    0   No   No
    "mm1": "sex",
    # MM2                    Survival status of sibling                      7176    1    N    I   20    0   No   No
    "mm2": "survival_status",
    # MM3                    Sibling's current age                           7196    2    N    I   20    0   No   No
    "mm3": "current_age",
    # MM4                    Sibling's date of birth (CMC)                   7236    4    N    I   20    0   No   No
    "mm4": "date_of_birth",
    # MM8                    Date of death of sibling (CMC)                  7416    4    N    I   20    0   No   No
    "mm8": "date_of_death",
    # MM7                    Sibling's age at death                          7376    2    N    I   20    0   No   No
    "mm7": "age_at_death",
    # MM9                    Sibling's death and pregnancy                   7496    2    N    I   20    0   No   No
    "mm9": "pregnancy_category",
    # MM16                   Sibling's death due to violence or accident     7816    1    N    I   20    0   No   No
    "mm16": "death_violence_or_accident",
}

In [ ]:
raw_wra_data = pd.read_stata(directory + "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA")
sibling_data = raw_wra_data[
    [
        c
        for c in raw_wra_data.columns
        if c in respondent_column_names.keys()
        or c.split("_")[0] in sibling_column_names.keys()
    ]
].copy()
sibling_data

In [ ]:
# inspired by https://stackoverflow.com/a/67393747/
sibling_data_reshaped = sibling_data[
    [c for c in sibling_data.columns if c.split("_")[0] in sibling_column_names.keys()]
].copy()
sibling_data_reshaped.columns = sibling_data_reshaped.columns.str.split(
    "_", expand=True
)
sibling_data_reshaped

In [ ]:
sibling_data_reshaped[list(respondent_column_names.keys())] = sibling_data[
    list(respondent_column_names.keys())
]
sibling_data_reshaped

In [ ]:
# Get a row per sibling
sibling_data_reshaped = (
    sibling_data_reshaped.set_index(list(respondent_column_names.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(respondent_column_names)}"])
)
sibling_data_reshaped

In [ ]:
sibling_data = (
    sibling_data_reshaped[
        list(respondent_column_names.keys()) + list(sibling_column_names.keys())
    ]
    .rename(columns=respondent_column_names)
    .rename(columns=sibling_column_names)
)
sibling_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(sibling_data.wealth_quintile)
sibling_data["weight"] = sibling_data.weight / 1_000_000
sibling_data

In [ ]:
# "A total of 219,561 siblings were recorded..." (p. 372)
len(sibling_data)

### Births

In [ ]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
}
if location == "india":
    birth_columns["s220a"] = "duration_of_pregnancy"
else:
    birth_columns["b20"] = "duration_of_pregnancy"

birth_data = pd.read_stata(
    directory
    + (
        "IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA"
        if location == "india"
        else "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA"
    ),
    columns=birth_columns.keys(),
)
birth_data

In [ ]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

### Maternal mortality ratio

#### Maternal mortality rate

In [ ]:
sibling_data.survival_status.value_counts()

In [ ]:
sibling_data.pregnancy_category.value_counts()

In [ ]:
sibling_data.death_violence_or_accident.value_counts()

In [ ]:
# https://dhsprogram.com/Data/Guide-to-DHS-Statistics/Adult_Mortality_Rates.htm#Calculation1
sibling_data["exposure_start"] = np.maximum(
    sibling_data.date_of_birth + 12 * 15, sibling_data.interview_date - 84
)  # aka lowlim
# aka upplim
sibling_data["exposure_end"] = np.minimum(
    np.where(
        sibling_data.survival_status == "alive",
        sibling_data.interview_date - 1,
        sibling_data.date_of_death,
    ),
    sibling_data.date_of_birth + 12 * 50 - 1,
)
sibling_data["exposure"] = (
    (sibling_data.exposure_end - sibling_data.exposure_start) + 1
).clip(lower=0)

In [ ]:
sibling_data.exposure.value_counts()

In [ ]:
sibling_data["adult_death"] = (
    (sibling_data.survival_status == "dead")
    & (sibling_data.date_of_death - sibling_data.date_of_birth >= 15.0 * 12)
    & (sibling_data.date_of_death - sibling_data.date_of_birth < 50.0 * 12)
    & (sibling_data.exposure > 0)
    & (sibling_data.date_of_death >= sibling_data.exposure_start)
    & (sibling_data.date_of_death <= sibling_data.exposure_end)
)

In [ ]:
# Matches table 14.2
(
    sibling_data[sibling_data.date_of_birth.notnull()]
    .assign(weighted_exposure=lambda df: df.exposure * df.weight)
    .groupby("sex")
    .weighted_exposure.sum()
    / 12
)

In [ ]:
# Matches table 14.2
sibling_data.assign(
    weighted_adult_dealth=lambda df: df.adult_death * df.weight
).groupby("sex").weighted_adult_dealth.sum()

In [ ]:
sibling_data.death_violence_or_accident.value_counts()

In [ ]:
sibling_data.pregnancy_category.value_counts()

In [ ]:
female_siblings = sibling_data[sibling_data.sex == "female"].copy()
female_siblings["maternal_death"] = (
    (female_siblings.adult_death)
    & (
        female_siblings.pregnancy_category.isin(
            ["died during delivery", "died while pregnant", "6 weeks after delivery"]
        )
    )
    & (~female_siblings.death_violence_or_accident.isin(["violence", "accident"]))
)

In [ ]:
# Table 14.4 reports 451 maternal deaths
(female_siblings.maternal_death * female_siblings.weight).sum()

In [ ]:
# Table 14.4 reports 480,382
(female_siblings.exposure * female_siblings.weight / 12).sum()

In [ ]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        (df.exposure * df.weight) / 12
    ).sum()

In [ ]:
# "the maternal mortality rate among women age 15-49 is 0.92 deaths per 1,000 woman-years of exposure." (p. 374)
# TODO: These are not age-standardized! We figure the *disparity* probably isn't way off.
# Should standardize according to the approach from https://github.com/LateraOlana/Fertility_SIM_DHS/blob/main/fertility/Latera_Zebb_Coworking.ipynb
maternal_mortality_rate(female_siblings)

In [ ]:
maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

#### General fertility rate

In [ ]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [ ]:
fertility_event_data.weighted_birth_in_period.sum()

In [ ]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

In [ ]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [ ]:
# Within rounding error of value reported in Table 5.1
fertility_event_data.weighted_birth_in_period.sum() * 1_000 / (
    fertility_exposure_data.weighted_exposure.sum() / 12
)

In [ ]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

In [ ]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

In [ ]:
maternal_disorders_incidence_disparities.to_csv(
    f"{results_dir}/maternal_disorders_incidence_disparities/nigeria.csv",
    index=False,
)